# CNN Gender Classifier

## Clasificación binaria de rostros usando una red neuronal convolucional desde cero

Este notebook implementa el flujo completo de entrenamiento de una red neuronal convolucional para clasificar imágenes de rostros en dos clases:

- `male`
- `female`

El objetivo es construir una CNN desde cero usando TensorFlow/Keras, entrenarla con imágenes RGB, evaluar su desempeño y guardar el modelo final en formato `.keras` para posteriormente desplegarlo como una API independiente.

El proyecto incluye:

1. Exploración del dataset.
2. Preprocesamiento de imágenes.
3. División en entrenamiento, validación y prueba.
4. Construcción de una CNN desde cero.
5. Entrenamiento del modelo.
6. Evaluación con métricas de clasificación.
7. Visualización de curvas de entrenamiento.
8. Interpretabilidad visual con Grad-CAM y Saliency Map.
9. Exportación del modelo para despliegue.

# 1. Configuración inicial

En esta sección se configuran las librerías, rutas globales, semillas de reproducibilidad y parámetros principales del proyecto.

El backend que consumirá este modelo espera que el entrenamiento use:

- Imágenes en formato RGB.
- Tamaño uniforme de `224x224`.
- Valores de píxeles normalizados en el rango `[0, 1]`.
- Salida binaria con activación sigmoide.
- Modelo guardado en `models/model.keras`.

Es fundamental que el preprocesamiento usado durante el entrenamiento sea exactamente el mismo que se aplicará durante la inferencia en la API.

In [ ]:
# Instalación de dependencias principales.
# En Google Colab muchas de estas librerías ya vienen instaladas,
# pero esta celda ayuda a asegurar compatibilidad.

!pip install -q tensorflow scikit-learn matplotlib pillow opencv-python kaggle

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# Verificamos versiones principales del entorno.

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

# 2. Reproducibilidad

Para que los resultados sean más estables entre ejecuciones, se fija una semilla global.

Esto no garantiza resultados idénticos en todos los equipos, especialmente si se usa GPU, pero sí reduce la variabilidad del entrenamiento.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

print("Semilla configurada:", SEED)

# 3. Rutas del proyecto

Se definen las rutas principales del proyecto.

La estructura esperada será:

```txt
cnn-gender-classifier/
│
├── backend/
│   └── models/
│       └── model.keras
│
├── data/
│   ├── male/
│   └── female/
│
├── notebooks/
│   └── train_cnn.ipynb
│
├── plots/
│
└── reports/

In [ ]:
# Detectar ruta base del proyecto.
# Si el notebook está dentro de /notebooks, subimos un nivel.
# Si está en la raíz del proyecto, usamos la carpeta actual.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
MALE_DIR = DATA_DIR / "male"
FEMALE_DIR = DATA_DIR / "female"

BACKEND_DIR = PROJECT_ROOT / "backend"
MODELS_DIR = BACKEND_DIR / "models"
MODEL_PATH = MODELS_DIR / "model.keras"

PLOTS_DIR = PROJECT_ROOT / "plots"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Crear carpetas necesarias si no existen
for directory in [DATA_DIR, MALE_DIR, FEMALE_DIR, MODELS_DIR, PLOTS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("PLOTS_DIR:", PLOTS_DIR)

# 4. Parámetros globales del experimento

Se definen los hiperparámetros principales que se reutilizarán durante todo el notebook.

Estos valores deben mantenerse consistentes con el backend:

- `IMG_SIZE`: tamaño esperado por la CNN.
- `BATCH_SIZE`: número de imágenes procesadas por lote.
- `EPOCHS`: número inicial de épocas de entrenamiento.
- `LEARNING_RATE`: tasa de aprendizaje del optimizador.
- `CLASS_NAMES`: nombres de las clases del problema.

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)

BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4

CLASS_NAMES = ["female", "male"]

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Class names:", CLASS_NAMES)

# 5. Verificación inicial de carpetas

Antes de cargar el dataset, se verifica si las carpetas `data/male` y `data/female` existen y si contienen imágenes.

En este punto todavía no es obligatorio que el dataset esté descargado. Esta celda solo nos ayuda a confirmar el estado actual del proyecto.

In [ ]:
def count_files(directory: Path):
    image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    
    if not directory.exists():
        return 0
    
    return len([
        file for file in directory.rglob("*")
        if file.suffix.lower() in image_extensions
    ])


male_count = count_files(MALE_DIR)
female_count = count_files(FEMALE_DIR)

print("Imágenes en male:", male_count)
print("Imágenes en female:", female_count)

if male_count == 0 or female_count == 0:
    print("\nEl dataset aún no parece estar cargado completamente.")
    print("En las siguientes celdas se realizará la descarga o se indicará cómo ubicarlo.")
else:
    print("\nDataset detectado correctamente.")